# 03 · Fine-Tuning com MultipleNegativesRankingLoss

Ajusta o modelo pré-treinado ao domínio do WikiIR 1k.

**Loss:** `MultipleNegativesRankingLoss`
- Usa pares `(query, doc_positivo)`
- Demais exemplos no batch atuam como **negativos implícitos**
- Estado-da-arte para tarefas de retrieval denso

## 1 · Instalação

In [1]:
#!py -m pip install sentence-transformers torch

## 2 · Imports e configuração

In [2]:
import json, math
from pathlib import Path
import torch
from tqdm import tqdm
from sentence_transformers import (
    InputExample, SentenceTransformer,
    losses, evaluation,
)
from torch.utils.data import DataLoader

BASE_MODEL   = 'sentence-transformers/all-MiniLM-L6-v2'
OUTPUT_DIR   = Path('models/finetuned')
DATA_DIR     = Path('data')
EPOCHS       = 10
BATCH_SIZE   = 32
WARMUP_RATIO = 0.1
LR           = 2e-5
EVAL_STEPS   = 200
MAX_SEQ_LEN  = 256

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device disponível: {device}')

Device disponível: cuda


C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_15220\751149447.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_15220\751149447.py:5: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import (


## 3 · Carregamento dos exemplos de treino

In [ ]:
def load_examples(path, use_triplets=False):
    examples = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            if use_triplets and 'negative' in item:
                examples.append(InputExample(
                    texts=[item['query'], item['positive'], item['negative']]))
            else:
                examples.append(InputExample(
                    texts=[item['query'], item['positive']]))
    return examples

train_examples = load_examples(DATA_DIR / 'train_hard_triplets.jsonl', use_triplets=True)
print(f'Exemplos de treino: {len(train_examples):,}')
print(f'Usando hard negatives: {len(train_examples[0].texts) == 3}')

## 4 · Modelo base

In [4]:
print(f'Carregando modelo: {BASE_MODEL}')
model = SentenceTransformer(BASE_MODEL, device=device)
model.max_seq_length = MAX_SEQ_LEN
print('Pronto!')

Carregando modelo: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pronto!


## 5 · DataLoader e Loss

In [5]:
train_loader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
    drop_last=True,
    collate_fn=lambda batch: batch,  # retorna lista de InputExample
)

train_loss = losses.MultipleNegativesRankingLoss(model)
print(f'Loss: MultipleNegativesRankingLoss | {len(train_loader)} steps/epoch')

Loss: MultipleNegativesRankingLoss | 121 steps/epoch


## 6 · Avaliador de IR (InformationRetrievalEvaluator)

In [6]:
def build_ir_evaluator(data_dir):
    queries, corpus, relevant_docs = {}, {}, {}

    with open(data_dir / 'queries.jsonl') as f:
        for line in f:
            item = json.loads(line)
            queries[item['query_id']] = item['text']

    with open(data_dir / 'corpus.jsonl') as f:
        for line in f:
            item = json.loads(line)
            corpus[item['doc_id']] = item['text']

    relevant_docs = {qid: set() for qid in queries}
    with open(data_dir / 'qrels.txt') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 4:
                qid, _, did, rel = parts
                if qid in relevant_docs and int(rel) >= 1:
                    relevant_docs[qid].add(did)

    relevant_docs = {k: v for k, v in relevant_docs.items() if v}

    return evaluation.InformationRetrievalEvaluator(
        queries=queries, corpus=corpus, relevant_docs=relevant_docs,
        name='wikiir-val',
        mrr_at_k=[10], ndcg_at_k=[10], map_at_k=[10, 100],
        show_progress_bar=False,
    )

evaluator = build_ir_evaluator(DATA_DIR)
print('Avaliador criado!')

Avaliador criado!


## 7 · Configuração do scheduler

In [7]:
total_steps  = len(train_loader) * EPOCHS
warmup_steps = math.ceil(total_steps * WARMUP_RATIO)

print(f'Steps totais : {total_steps:,}')
print(f'Warmup steps : {warmup_steps}')
print(f'Epochs       : {EPOCHS}')
print(f'Batch size   : {BATCH_SIZE}')
print(f'Learning rate: {LR}')

Steps totais : 1,210
Warmup steps : 121
Epochs       : 10
Batch size   : 32
Learning rate: 2e-05


## 8 · Fine-Tuning

> ⚠️ **Tempo estimado:** ~30 min (GPU) / 2–4 h (CPU)  
> Para testes rápidos, reduza `EPOCHS = 1`.

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

def primary_score(metrics):
    key = [k for k in metrics if 'ndcg' in k.lower()][0]
    return metrics[key]

use_hard_negs = len(train_examples[0].texts) == 3
print(f'Hard negatives: {use_hard_negs}')

best_score = -1
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        queries   = [ex.texts[0] for ex in batch]
        positives = [ex.texts[1] for ex in batch]

        feat_q = {k: v.to(device) for k, v in model.tokenize(queries).items() if isinstance(v, torch.Tensor)}
        feat_p = {k: v.to(device) for k, v in model.tokenize(positives).items() if isinstance(v, torch.Tensor)}
        features = [feat_q, feat_p]

        if use_hard_negs:
            hard_negs = [ex.texts[2] for ex in batch]
            feat_n = {k: v.to(device) for k, v in model.tokenize(hard_negs).items() if isinstance(v, torch.Tensor)}
            features.append(feat_n)

        optimizer.zero_grad()
        with torch.autocast(device_type=device, enabled=(device == 'cuda')):
            loss_val = train_loss(features, None)

        scaler.scale(loss_val).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss += loss_val.item()
        global_step += 1

        if global_step % EVAL_STEPS == 0:
            metrics = evaluator(model, output_path=str(OUTPUT_DIR))
            score = primary_score(metrics)
            print(f'Step {global_step} | nDCG@10={score:.4f} | best={best_score:.4f}')
            if score > best_score:
                best_score = score
                model.save(str(OUTPUT_DIR))
                print('  → Melhor modelo salvo!')
            model.train()

    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} | loss={avg_loss:.4f}')

metrics = evaluator(model, output_path=str(OUTPUT_DIR))
score = primary_score(metrics)
print(f'\nMétricas finais:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')
if score > best_score:
    model.save(str(OUTPUT_DIR))

print(f'\nFine-tuning concluído! Modelo salvo em {OUTPUT_DIR}')

## 9 · Verificação do modelo salvo

In [10]:
from sentence_transformers import SentenceTransformer as ST

ft_model = ST(str(OUTPUT_DIR))
test_emb = ft_model.encode(['teste de embedding'])
print(f'Modelo fine-tuned carregado com sucesso!')
print(f'Dimensão do embedding: {test_emb.shape[1]}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo fine-tuned carregado com sucesso!
Dimensão do embedding: 384
